# CBR-RAG Evaluation on PubMedQA using Gemma

This notebook runs:
- `no_rag`
- `keyword_rag`
- `semantic_rag`
- `cbr_rag`

using the fixed evaluation IDs from `test_ids_250.json`.


In [6]:
!pip install biopython sentence-transformers faiss-cpu requests python-dotenv --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 86.0 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import json
import os
import re
import sys
import textwrap
from collections import Counter

import numpy as np
from Bio import Entrez
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# Add the directory containing the modules to sys.path
sys.path.append('/content/drive/MyDrive/Colab Notebooks/NLP Notebooks')
from cbr_retrieval import PubMedQACaseRetriever

In [ ]:
import os

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

MODEL_NAME = "google/gemma-3-12b-it"

TOP_K_DOCS = 6
SEMANTIC_CANDIDATES = 40
GRAPH_SEED_K = 25
CBR_TOP_CASES = 4
GRAPH_BUILD_LIMIT = None
CASE_BUILD_LIMIT = None

Entrez.email = "rithvikhariprasad@gmail.com"
Entrez.tool = "pubmed-cbr-rag"

QUESTIONS_FILE = "/content/drive/MyDrive/ori_pqal.json"
GROUND_TRUTH_FILE = "/content/drive/MyDrive/test_ground_truth.json"
#TEST_IDS_FILE = "/content/drive/MyDrive/test_ids_250.json"

if not OPENROUTER_API_KEY:
    raise ValueError("Set OPENROUTER_API_KEY before running this notebook.")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

emb_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

pubmed_search_cache = {}
graph_retriever = None
case_retriever = None



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
test = client.chat.completions.create(
    model="google/gemma-3-12b-it",
    messages=[{"role": "user", "content": "Reply with only: working"}],
    max_tokens=5,
    temperature=0.0,
)
print(test.choices[0].message.content)


working



In [ ]:
def extract_answer(llm_output):
    if not llm_output:
        return "unknown"
    match = re.search(r"<answer>(.*?)</answer>", llm_output, flags=re.IGNORECASE | re.DOTALL)
    if match:
        ans = match.group(1).strip().lower()
        ans = re.sub(r"[^a-z]", "", ans)
        if ans in ["yes", "no", "maybe"]:
            return ans
    return "unknown"

def _normalize_embeddings(matrix):
    matrix = matrix.astype("float32")
    matrix /= (np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-12)
    return matrix

def _doc_text(doc):
    return f"{doc.get('title', '')} {doc.get('abstract', '')}".strip()

def pubmed_keyword_search(query: str, k: int = 5, verbose: bool = False):
    cache_key = (query, k)
    if cache_key in pubmed_search_cache:
        return [dict(doc) for doc in pubmed_search_cache[cache_key]]

    h = Entrez.esearch(db="pubmed", term=query, retmax=k)
    r = Entrez.read(h)
    h.close()
    pmids = r.get("IdList", [])
    if not pmids:
        pubmed_search_cache[cache_key] = []
        return []

    h = Entrez.efetch(db="pubmed", id=",".join(pmids), rettype="abstract", retmode="xml")
    recs = Entrez.read(h)
    h.close()
    if verbose:
        print("ESearch Count:", r["Count"], "Returning:", len(r["IdList"]))

    docs = []
    for article in recs["PubmedArticle"]:
        art = article["MedlineCitation"]["Article"]
        pmid = str(article["MedlineCitation"]["PMID"])
        title = str(art.get("ArticleTitle", ""))
        abstract_list = art.get("Abstract", {}).get("AbstractText", [])
        abstract = " ".join(str(x) for x in abstract_list)
        docs.append({"pmid": pmid, "title": title, "abstract": abstract})

    pubmed_search_cache[cache_key] = [dict(doc) for doc in docs]
    return docs

def semantic_topk_from_pubmed(query: str, k: int = 5, n_candidates: int = 40):
    candidates = pubmed_keyword_search(query, k=n_candidates)
    if not candidates:
        return []

    texts = [_doc_text(d) for d in candidates]
    q_emb = _normalize_embeddings(emb_model.encode([query], convert_to_numpy=True))
    d_emb = _normalize_embeddings(emb_model.encode(texts, convert_to_numpy=True))

    semantic_scores = d_emb @ q_emb[0]
    lexical_scores = np.array([lexical_overlap_score(query, text) for text in texts], dtype="float32")
    final_scores = 0.85 * semantic_scores + 0.15 * lexical_scores
    order = np.argsort(-final_scores)[:k]

    results = []
    for idx in order:
        d = dict(candidates[int(idx)])
        d["score"] = float(final_scores[int(idx)])
        d["semantic_score"] = float(semantic_scores[int(idx)])
        d["lexical_score"] = float(lexical_scores[int(idx)])
        results.append(d)
    return results

def graph_topk_from_pubmedqa(query: str, k: int = 5, seed_k: int = 25):
    if graph_retriever is None:
        raise ValueError("Build the graph index first.")
    return graph_retriever.retrieve(query, k=k, seed_k=seed_k)

def cbr_topk_from_pubmedqa(query: str, k_cases: int = 4):
    if case_retriever is None:
        raise ValueError("Build the case base first.")
    return case_retriever.retrieve(query, k=k_cases)

def build_context(docs, max_chars=6000):
    parts = []
    for i, d in enumerate(docs, start=1):
        snippet = d["abstract"][:1200]
        parts.append(f"[{i}] PMID: {d['pmid']}\nTitle: {d['title']}\nAbstract: {snippet}\n")
    return "\n\n".join(parts)[:max_chars]

def build_case_context(cases, max_chars=3500):
    parts = []
    for i, case in enumerate(cases, start=1):
        rationale = (case.get("long_answer") or "").strip()
        if rationale:
            rationale = rationale[:500]
        elif case.get("contexts"):
            rationale = " ".join(case["contexts"][:2])[:500]
        else:
            rationale = "No rationale available."
        parts.append(
            f"[Case {i}] Similar question: {case['question']}\n"
            f"Final answer: {case['final_decision']}\n"
            f"Reasoning pattern: {rationale}\n"
        )
    return "\n\n".join(parts)[:max_chars]

def build_rag_prompts(user_query, docs):
    ctx = build_context(docs)
    system_msg = (
        "You are a careful biomedical question answering assistant. "
        "Use only the provided evidence. "
        "Answer 'yes' only when the evidence clearly supports the claim. "
        "Answer 'no' when the evidence clearly contradicts the claim or fails to support it. "
        "Answer 'maybe' only when the evidence is genuinely mixed or insufficient."
    )
    user_msg = textwrap.dedent(f"""Question: {user_query}

    PubMed sources:
    {ctx}

    When answering:
    - Think briefly inside <thinking> tags.
    - Then output exactly one final label in <answer> tags: yes, no, or maybe.
    - Prefer 'no' over 'maybe' when the evidence does not support the claim.
    - Do not invent facts not present in the sources.""")
    return system_msg, user_msg

def build_cbr_prompts(user_query, docs, cases):
    evidence_ctx = build_context(docs, max_chars=4500)
    case_ctx = build_case_context(cases, max_chars=2500)
    system_msg = (
        "You are a careful biomedical question answering assistant using case-based reasoning. "
        "Use similar past PubMedQA cases as reasoning patterns only, and use the current PubMed evidence as the factual basis for the final answer. "
        "Do not copy a past answer unless the current evidence supports it."
    )
    user_msg = textwrap.dedent(f"""Target question: {user_query}

    Similar past QA cases:
    {case_ctx}

    Current PubMed evidence:
    {evidence_ctx}

    Instructions:
    - First compare the target question to the similar past cases inside <thinking> tags.
    - Use the current PubMed evidence to verify whether the same reasoning applies.
    - Then output exactly one final label in <answer> tags: yes, no, or maybe.
    - If the past cases conflict with the current evidence, trust the current evidence.
    - Prefer 'no' over 'maybe' when the evidence does not support the claim.
    - Do not invent facts not present in the provided cases or evidence.""")
    return system_msg, user_msg

def call_model(system_prompt, user_prompt, max_tokens=256):
    completion = client.chat.completions.create(
        extra_headers={
            "HTTP-Referer": "<YOUR_SITE_URL>",
            "X-OpenRouter-Title": "<YOUR_SITE_NAME>",
        },
        extra_body={},
        model=MODEL_NAME,
        temperature=0.0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return completion.choices[0].message.content


In [ ]:

print("Loading datasets...")
with open(QUESTIONS_FILE, "r", encoding="utf-8") as f:
    questions_data = json.load(f)

with open(GROUND_TRUTH_FILE, "r", encoding="utf-8") as f:
    gt_data = json.load(f)


test_ids = [
    str(q_id)
    for q_id in gt_data.keys()
    if str(q_id) in questions_data
]

print(f"Loaded {len(test_ids)} test IDs from ground truth file")

label_counts = Counter()
for q_id in test_ids:
    gt_entry = gt_data[q_id]
    if isinstance(gt_entry, dict):
        label = gt_entry.get("final_decision", "").strip().lower()
    else:
        label = str(gt_entry).strip().lower()
    label_counts[label] += 1
print("Sample label counts:", dict(label_counts))

print("Building PubMedQA graph index...")
graph_retriever = PubMedQAGraphRetriever(
    emb_model=emb_model,
    graph_k=10,
    diffusion_alpha=0.60,
    diffusion_steps=8,
    seed_k=GRAPH_SEED_K,
).fit(questions_data, max_questions=GRAPH_BUILD_LIMIT)
print(f"Graph index ready with {len(graph_retriever.documents)} context nodes.")

print("Building PubMedQA case base for CBR-RAG...")
case_retriever = PubMedQACaseRetriever(emb_model=emb_model).fit(
    questions_data,
    gt_data=gt_data,
    excluded_ids=test_ids,
    max_cases=CASE_BUILD_LIMIT,
)
print(f"Case base ready with {len(case_retriever.cases)} train cases.")


Loading datasets...
Loaded 500 test IDs from ground truth file
Sample label counts: {'yes': 276, 'no': 169, 'maybe': 55}
Building PubMedQA graph index...


Batches:   0%|          | 0/105 [00:00<?, ?it/s]

Graph index ready with 3358 context nodes.
Building PubMedQA case base for CBR-RAG...
Case base ready with 500 train cases.


In [ ]:
import os
import ast
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

LABELS = ["yes", "no", "maybe"]

CHECKPOINT_PATH = "/content/drive/MyDrive/rag_checkpoint_cbr500.csv"
SAVE_EVERY = 50

def get_ground_truth(q_id):
    gt_entry = gt_data[q_id]
    if isinstance(gt_entry, dict):
        return gt_entry.get("final_decision", "").strip().lower()
    return str(gt_entry).strip().lower()


def get_pmids_from_docs(docs):
    pmids = []
    for d in docs:
        if isinstance(d, dict):
            pmid = d.get("pmid") or d.get("pubmed_id") or d.get("PMID")
            if pmid is not None:
                pmids.append(str(pmid))
    return pmids


def add_prediction(all_rows, q_id, method, query, gold, pred, docs=None, error=None):
    docs = docs or []
    all_rows.append({
        "qid": str(q_id),
        "method": method,
        "question": query,
        "gold": gold,
        "pred": pred,
        "retrieved_pmids": get_pmids_from_docs(docs),
        "error": error
    })


# Resume checkpoint
if os.path.exists(CHECKPOINT_PATH):
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH)

    # convert retrieved_pmids back from string to list
    if "retrieved_pmids" in checkpoint_df.columns:
        checkpoint_df["retrieved_pmids"] = checkpoint_df["retrieved_pmids"].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else []
        )

    all_predictions = checkpoint_df.to_dict("records")

    # done only when all selected methods exist for that qid
    done_ids = set(checkpoint_df["qid"].astype(str).unique())

    print(f"▶️ Resuming from checkpoint — {len(done_ids)} question IDs already attempted.")
else:
    all_predictions = []
    done_ids = set()
    print("🆕 Starting fresh.")

▶️ Resuming from checkpoint — 250 question IDs already attempted.


In [ ]:
# =========================
# MAIN EVALUATION LOOP WITH CHECKPOINTING
# =========================

failed_qids = []

for i, q_id in enumerate(tqdm(test_ids, desc="Evaluating RAG Pipelines")):
    q_id = str(q_id)

    if q_id in done_ids:
        continue

    try:
        query = questions_data[q_id].get("QUESTION", "")
        ground_truth = get_ground_truth(q_id)

        # Skip invalid labels
        if ground_truth not in LABELS:
            failed_qids.append((q_id, f"Invalid gold label: {ground_truth}"))
            continue

        # ---------- No RAG ----------
        ans_no_rag = extract_answer(
            call_model(*build_rag_prompts(query, []))
        )
        add_prediction(
            all_predictions, q_id, "no_rag", query, ground_truth, ans_no_rag, docs=[]
        )

        # ---------- Keyword RAG ----------
        docs_keyword = pubmed_keyword_search(query, k=TOP_K_DOCS)
        ans_kw = extract_answer(
            call_model(*build_rag_prompts(query, docs_keyword))
        )
        add_prediction(
            all_predictions, q_id, "keyword_rag", query, ground_truth, ans_kw, docs=docs_keyword
        )

        # ---------- Semantic RAG ----------
        docs_semantic = semantic_topk_from_pubmed(
            query,
            k=TOP_K_DOCS,
            n_candidates=SEMANTIC_CANDIDATES
        )
        ans_sem = extract_answer(
            call_model(*build_rag_prompts(query, docs_semantic))
        )
        add_prediction(
            all_predictions, q_id, "semantic_rag", query, ground_truth, ans_sem, docs=docs_semantic
        )

        # ---------- CBR RAG ----------
        cbr_cases = cbr_topk_from_pubmedqa(query, k_cases=CBR_TOP_CASES)
        cbr_docs = docs_semantic if docs_semantic else docs_keyword
        ans_cbr = extract_answer(
            call_model(*build_cbr_prompts(query, cbr_docs, cbr_cases))
        )

        # For retrieved_pmids, store both evidence docs and CBR cases if possible
        combined_docs = []
        if cbr_docs:
            combined_docs.extend(cbr_docs)
        if cbr_cases:
            combined_docs.extend(cbr_cases)

        add_prediction(
            all_predictions, q_id, "cbr_rag", query, ground_truth, ans_cbr, docs=combined_docs
        )

        done_ids.add(q_id)

        print(f"\n--- Question ID: {q_id} ---")
        print(f"Ground Truth:   {ground_truth}")
        print(f"No RAG Output:  {ans_no_rag}")
        print(f"Keyword RAG:    {ans_kw}")
        print(f"Semantic RAG:   {ans_sem}")
        print(f"CBR RAG:        {ans_cbr}")

        # Save checkpoint
        if len(done_ids) % SAVE_EVERY == 0:
            temp_df = pd.DataFrame(all_predictions)
            temp_df.to_csv(CHECKPOINT_PATH, index=False)
            print(f"💾 Checkpoint saved at {len(done_ids)} completed questions.")

    except Exception as e:
        failed_qids.append((q_id, str(e)))
        print(f"\nSkipped Question ID {q_id} due to error: {e}")

        # Save even after error
        temp_df = pd.DataFrame(all_predictions)
        temp_df.to_csv(CHECKPOINT_PATH, index=False)

# Final save
pred_df = pd.DataFrame(all_predictions)
pred_df.to_csv(CHECKPOINT_PATH, index=False)

print(f"\nFinal checkpoint saved: {CHECKPOINT_PATH}")
print(f"Total prediction rows: {len(pred_df)}")
print(f"Total question IDs completed: {len(done_ids)}")

Evaluating RAG Pipelines:   1%|          | 3/500 [00:09<27:29,  3.32s/it]


--- Question ID: 19100463 ---
Ground Truth:   yes
No RAG Output:  maybe
Keyword RAG:    maybe
Semantic RAG:   maybe
CBR RAG:        maybe


Evaluating RAG Pipelines:   1%|          | 5/500 [00:21<38:04,  4.62s/it]


--- Question ID: 12913878 ---
Ground Truth:   yes
No RAG Output:  yes
Keyword RAG:    yes
Semantic RAG:   yes
CBR RAG:        yes


Evaluating RAG Pipelines:   1%|▏         | 7/500 [00:29<34:41,  4.22s/it]

...


--- Question ID: 20736672 ---
Ground Truth:   maybe
No RAG Output:  unknown
Keyword RAG:    yes
Semantic RAG:   yes
CBR RAG:        yes


Evaluating RAG Pipelines:  99%|█████████▉| 496/500 [1:02:40<00:26,  6.64s/it]


--- Question ID: 17691856 ---
Ground Truth:   maybe
No RAG Output:  maybe
Keyword RAG:    maybe
Semantic RAG:   maybe
CBR RAG:        no


Evaluating RAG Pipelines: 100%|██████████| 500/500 [1:02:51<00:00,  7.54s/it]


--- Question ID: 19694846 ---
Ground Truth:   maybe
No RAG Output:  unknown
Keyword RAG:    yes
Semantic RAG:   yes
CBR RAG:        yes
💾 Checkpoint saved at 500 completed questions.

✅ Final checkpoint saved: /content/drive/MyDrive/rag_checkpoint_cbr500.csv
Total prediction rows: 2000
Total question IDs completed: 500


In [ ]:
# =========================
# CLASSIFICATION METRICS
# =========================

pred_df = pd.read_csv(CHECKPOINT_PATH)

# Convert retrieved_pmids from string to list
pred_df["retrieved_pmids"] = pred_df["retrieved_pmids"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else []
)

summary_rows = []

for method in sorted(pred_df["method"].unique()):
    sub = pred_df[pred_df["method"] == method].copy()

    valid = sub[
        sub["gold"].isin(LABELS) &
        sub["pred"].isin(LABELS)
    ].copy()

    if len(valid) == 0:
        continue

    summary_rows.append({
        "Method": method,
        "Accuracy": accuracy_score(valid["gold"], valid["pred"]),
        "Macro F1": f1_score(valid["gold"], valid["pred"], average="macro"),
        "Macro Precision": precision_score(valid["gold"], valid["pred"], average="macro", zero_division=0),
        "Macro Recall": recall_score(valid["gold"], valid["pred"], average="macro", zero_division=0),
        "Valid N": len(valid),
        "Unknown/Invalid Outputs": len(sub) - len(valid),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

,Method,Accuracy,Macro F1,Macro Precision,Macro Recall,Valid N,Unknown/Invalid Outputs
0,cbr_rag,0.597590,0.485290,0.490901,0.488334,415,85
1,keyword_rag,0.496868,0.432829,0.474942,0.435167,479,21
2,no_rag,0.154519,0.147405,0.332658,0.333185,343,157
3,semantic_rag,0.494824,0.423593,0.476904,0.419017,483,17


In [ ]:
# =========================
# CLASSIFICATION REPORTS + CONFUSION MATRICES
# =========================

for method in sorted(pred_df["method"].unique()):
    sub = pred_df[pred_df["method"] == method].copy()

    valid = sub[
        sub["gold"].isin(LABELS) &
        sub["pred"].isin(LABELS)
    ].copy()

    if len(valid) == 0:
        print(f"\n{method.upper()}: No valid predictions")
        continue

    print("\n" + "=" * 80)
    print(method.upper())
    print("=" * 80)

    print(classification_report(
        valid["gold"],
        valid["pred"],
        labels=LABELS,
        zero_division=0
    ))

    cm = confusion_matrix(
        valid["gold"],
        valid["pred"],
        labels=LABELS
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{l}" for l in LABELS],
        columns=[f"pred_{l}" for l in LABELS]
    )

    display(cm_df)


CBR_RAG
              precision    recall  f1-score   support

         yes       0.75      0.64      0.69       231
          no       0.61      0.66      0.64       142
       maybe       0.10      0.17      0.13        42

    accuracy                           0.60       415
   macro avg       0.49      0.49      0.49       415
weighted avg       0.64      0.60      0.62       415



,pred_yes,pred_no,pred_maybe
true_yes,147,43,41
true_no,29,94,19
true_maybe,19,16,7



KEYWORD_RAG
              precision    recall  f1-score   support

         yes       0.70      0.58      0.63       261
          no       0.62      0.43      0.51       164
       maybe       0.11      0.30      0.16        54

    accuracy                           0.50       479
   macro avg       0.47      0.44      0.43       479
weighted avg       0.60      0.50      0.54       479



,pred_yes,pred_no,pred_maybe
true_yes,152,31,78
true_no,40,70,54
true_maybe,26,12,16



NO_RAG
              precision    recall  f1-score   support

         yes       0.60      0.02      0.03       192
          no       0.28      0.17      0.21       113
       maybe       0.11      0.82      0.20        38

    accuracy                           0.15       343
   macro avg       0.33      0.33      0.15       343
weighted avg       0.44      0.15      0.11       343



,pred_yes,pred_no,pred_maybe
true_yes,3,42,147
true_no,1,19,93
true_maybe,1,6,31



SEMANTIC_RAG
              precision    recall  f1-score   support

         yes       0.71      0.61      0.65       265
          no       0.63      0.39      0.48       164
       maybe       0.09      0.26      0.13        54

    accuracy                           0.49       483
   macro avg       0.48      0.42      0.42       483
weighted avg       0.61      0.49      0.54       483



,pred_yes,pred_no,pred_maybe
true_yes,161,25,79
true_no,39,64,61
true_maybe,28,12,14


In [ ]:
# =========================
# RETRIEVAL METRICS: HIT@K / RECALL@K
# =========================

def hit_at_k(qid, retrieved_pmids, k):
    retrieved_pmids = [str(x) for x in retrieved_pmids[:k]]
    return str(qid) in retrieved_pmids

def reciprocal_rank(qid, retrieved_pmids):
    qid = str(qid)
    retrieved_pmids = [str(x) for x in retrieved_pmids]
    for rank, pmid in enumerate(retrieved_pmids, start=1):
        if pmid == qid:
            return 1.0 / rank

    return 0.0

rag_methods = ["keyword_rag", "semantic_rag", "graph_rag", "cbr_rag"]
retrieval_rows = []
for method in rag_methods:
    sub = pred_df[pred_df["method"] == method].copy()
    if len(sub) == 0:
        continue
    row = {"Method": method}
    row["MRR"] = sub.apply(
        lambda r: reciprocal_rank(r["qid"], r["retrieved_pmids"]),
        axis=1
    ).mean()

    for k in [1, 3, 5, 10]:
        row[f"Hit@{k}"] = sub.apply(
            lambda r: hit_at_k(r["qid"], r["retrieved_pmids"], k),
            axis=1
        ).mean()
    retrieval_rows.append(row)

retrieval_summary_df = pd.DataFrame(retrieval_rows)

display(retrieval_summary_df)

,Method,MRR,Hit@1,Hit@3,Hit@5,Hit@10
0,keyword_rag,0.554633,0.488,0.602,0.660,0.670
1,semantic_rag,0.736733,0.714,0.756,0.766,0.766
2,cbr_rag,0.736733,0.714,0.756,0.766,0.766


In [ ]:
# =========================
# COMBINED FINAL RESULTS TABLE
# =========================

combined_results = summary_df.merge(
    retrieval_summary_df,
    on="Method",
    how="left"
)

display(combined_results)

,Method,Accuracy,Macro F1,Macro Precision,Macro Recall,Valid N,Unknown/Invalid Outputs,MRR,Hit@1,Hit@3,Hit@5,Hit@10
0,cbr_rag,0.597590,0.485290,0.490901,0.488334,415,85,0.736733,0.714,0.756,0.766,0.766
1,keyword_rag,0.496868,0.432829,0.474942,0.435167,479,21,0.554633,0.488,0.602,0.660,0.670
2,no_rag,0.154519,0.147405,0.332658,0.333185,343,157,NaN,NaN,NaN,NaN,NaN
3,semantic_rag,0.494824,0.423593,0.476904,0.419017,483,17,0.736733,0.714,0.756,0.766,0.766


Error Analysis

In [ ]:
# =========================
# ERROR ANALYSIS
# =========================

import ast
import pandas as pd

LABELS = ["yes", "no", "maybe"]

# Safety: convert retrieved_pmids from string to list if needed
def parse_pmids(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str) and x.startswith("["):
        try:
            return ast.literal_eval(x)
        except Exception:
            return []
    return []

pred_df["retrieved_pmids"] = pred_df["retrieved_pmids"].apply(parse_pmids)


# -------------------------
# Helper functions
# -------------------------
def gold_in_retrieved(row, k=None):
    retrieved = row["retrieved_pmids"]
    if k is not None:
        retrieved = retrieved[:k]

    retrieved = [str(x) for x in retrieved]
    return str(row["qid"]) in retrieved


def classify_error(row):
    gold = row["gold"]
    pred = row["pred"]

    # No-RAG has no retrieval step
    if row["method"] == "no_rag":
        if gold == pred:
            return "correct"
        elif pred not in LABELS:
            return "invalid_output"
        else:
            return "model_error_no_rag"

    # RAG methods
    if gold == pred:
        return "correct"

    if pred not in LABELS:
        return "invalid_output"

    if not gold_in_retrieved(row, k=None):
        return "retrieval_failure"

    return "generation_failure"


# -------------------------
# Add analysis columns
# -------------------------
pred_df["is_correct"] = pred_df["gold"] == pred_df["pred"]
pred_df["gold_in_top1"] = pred_df.apply(lambda r: gold_in_retrieved(r, k=1), axis=1)
pred_df["gold_in_top3"] = pred_df.apply(lambda r: gold_in_retrieved(r, k=3), axis=1)
pred_df["gold_in_top5"] = pred_df.apply(lambda r: gold_in_retrieved(r, k=5), axis=1)
pred_df["gold_in_top10"] = pred_df.apply(lambda r: gold_in_retrieved(r, k=10), axis=1)
pred_df["error_type"] = pred_df.apply(classify_error, axis=1)

display(pred_df.head())

,qid,method,question,gold,pred,retrieved_pmids,error,is_correct,gold_in_top1,gold_in_top3,gold_in_top5,gold_in_top10,error_type
0,10834864,no_rag,Risk factors for avascular necrosis of bone in...,no,unknown,[],NaN,False,False,False,False,False,invalid_output
1,10834864,keyword_rag,Risk factors for avascular necrosis of bone in...,no,unknown,[],NaN,False,False,False,False,False,invalid_output
2,10834864,semantic_rag,Risk factors for avascular necrosis of bone in...,no,unknown,[],NaN,False,False,False,False,False,invalid_output
3,10834864,cbr_rag,Risk factors for avascular necrosis of bone in...,no,maybe,[],NaN,False,False,False,False,False,retrieval_failure
4,11079675,no_rag,Pulmonary valve replacement in adults late aft...,yes,maybe,[],NaN,False,False,False,False,False,model_error_no_rag


In [ ]:
# =========================
# ERROR TYPE SUMMARY
# =========================

error_summary = (
    pred_df
    .groupby(["method", "error_type"])
    .size()
    .reset_index(name="count")
)

display(error_summary)

# Pivoted version: better for report/presentation
error_pivot = (
    error_summary
    .pivot(index="method", columns="error_type", values="count")
    .fillna(0)
    .astype(int)
    .reset_index()
)

display(error_pivot)


,method,error_type,count
0,cbr_rag,correct,248
1,cbr_rag,generation_failure,99
2,cbr_rag,invalid_output,85
3,cbr_rag,retrieval_failure,68
4,keyword_rag,correct,238
5,keyword_rag,generation_failure,147
6,keyword_rag,invalid_output,21
7,keyword_rag,retrieval_failure,94
8,no_rag,correct,53
9,no_rag,invalid_output,157


error_type,method,correct,generation_failure,invalid_output,model_error_no_rag,retrieval_failure
0,cbr_rag,248,99,85,0,68
1,keyword_rag,238,147,21,0,94
2,no_rag,53,0,157,290,0
3,semantic_rag,239,169,17,0,75


In [ ]:
# =========================
# ERROR TYPE PERCENTAGES
# =========================

error_percent = error_pivot.copy()

count_cols = [c for c in error_percent.columns if c != "method"]
error_percent["total"] = error_percent[count_cols].sum(axis=1)

for col in count_cols:
    error_percent[col + "_pct"] = error_percent[col] / error_percent["total"]

display(error_percent)


error_type,method,correct,generation_failure,invalid_output,model_error_no_rag,retrieval_failure,total,correct_pct,generation_failure_pct,invalid_output_pct,model_error_no_rag_pct,retrieval_failure_pct
0,cbr_rag,248,99,85,0,68,500,0.496,0.198,0.170,0.00,0.136
1,keyword_rag,238,147,21,0,94,500,0.476,0.294,0.042,0.00,0.188
2,no_rag,53,0,157,290,0,500,0.106,0.000,0.314,0.58,0.000
3,semantic_rag,239,169,17,0,75,500,0.478,0.338,0.034,0.00,0.150


In [ ]:
# =========================
# SHOW RETRIEVAL FAILURES
# =========================

def show_retrieval_failures(method="semantic_rag", n=5):
    sub = pred_df[
        (pred_df["method"] == method) &
        (pred_df["error_type"] == "retrieval_failure")
    ].copy()

    print(f"{method}: {len(sub)} retrieval failures")

    for _, row in sub.head(n).iterrows():
        print("\n" + "=" * 100)
        print("QID / Gold PMID:", row["qid"])
        print("Question:", row["question"])
        print("Gold:", row["gold"])
        print("Pred:", row["pred"])
        print("Retrieved PMIDs:", row["retrieved_pmids"])
        print("Gold in Top 1:", row["gold_in_top1"])
        print("Gold in Top 3:", row["gold_in_top3"])
        print("Gold in Top 5:", row["gold_in_top5"])

show_retrieval_failures("semantic_rag", n=5)

semantic_rag: 75 retrieval failures

QID / Gold PMID: 12121321
Question: Do mossy fibers release GABA?
Gold: yes
Pred: maybe
Retrieved PMIDs: ['23365214', '22915124', '23316138', '22842523', '24319410', '22711957']
Gold in Top 1: False
Gold in Top 3: False
Gold in Top 5: False

QID / Gold PMID: 20084845
Question: Biomolecular identification of allergenic pollen: a new perspective for aerobiological monitoring?
Gold: yes
Pred: maybe
Retrieved PMIDs: []
Gold in Top 1: False
Gold in Top 3: False
Gold in Top 5: False

QID / Gold PMID: 23690198
Question: Implementation of epidural analgesia for labor: is the standard of effective analgesia reachable in all women?
Gold: yes
Pred: maybe
Retrieved PMIDs: []
Gold in Top 1: False
Gold in Top 3: False
Gold in Top 5: False

QID / Gold PMID: 12163782
Question: Increased neutrophil migratory activity after major trauma: a factor in the etiology of acute respiratory distress syndrome?
Gold: yes
Pred: maybe
Retrieved PMIDs: []
Gold in Top 1: False
Gol

In [ ]:
# =========================
# SHOW GENERATION FAILURES
# =========================

def show_generation_failures(method="semantic_rag", n=5):
    sub = pred_df[
        (pred_df["method"] == method) &
        (pred_df["error_type"] == "generation_failure")
    ].copy()

    print(f"{method}: {len(sub)} generation failures")

    for _, row in sub.head(n).iterrows():
        print("\n" + "=" * 100)
        print("QID / Gold PMID:", row["qid"])
        print("Question:", row["question"])
        print("Gold:", row["gold"])
        print("Pred:", row["pred"])
        print("Retrieved PMIDs:", row["retrieved_pmids"])
        print("Gold in Top 1:", row["gold_in_top1"])
        print("Gold in Top 3:", row["gold_in_top3"])
        print("Gold in Top 5:", row["gold_in_top5"])

show_generation_failures("semantic_rag", n=5)

semantic_rag: 169 generation failures

QID / Gold PMID: 11079675
Question: Pulmonary valve replacement in adults late after repair of tetralogy of fallot: are we operating too late?
Gold: yes
Pred: maybe
Retrieved PMIDs: ['11079675', '4010322', '18290882']
Gold in Top 1: True
Gold in Top 3: True
Gold in Top 5: True

QID / Gold PMID: 18926458
Question: Are octogenarians at high risk for carotid endarterectomy?
Gold: no
Pred: maybe
Retrieved PMIDs: ['18926458', '19823705', '18091703', '25499530', '15768004', '27625000']
Gold in Top 1: True
Gold in Top 3: True
Gold in Top 5: True

QID / Gold PMID: 15943725
Question: Should serum pancreatic lipase replace serum amylase as a biomarker of acute pancreatitis?
Gold: yes
Pred: maybe
Retrieved PMIDs: ['15943725']
Gold in Top 1: True
Gold in Top 3: True
Gold in Top 5: True

QID / Gold PMID: 16735905
Question: Does the severity of obstructive sleep apnea predict patients requiring high continuous positive airway pressure?
Gold: maybe
Pred: yes
Ret

In [ ]:
# =========================
# CASES WHERE RAG HELPED
# =========================

def show_rag_helped(rag_method="semantic_rag", n=5):
    no_rag = pred_df[pred_df["method"] == "no_rag"][["qid", "pred"]].rename(
        columns={"pred": "no_rag_pred"}
    )

    rag = pred_df[pred_df["method"] == rag_method].copy()

    merged = rag.merge(no_rag, on="qid", how="inner")

    helped = merged[
        (merged["no_rag_pred"] != merged["gold"]) &
        (merged["pred"] == merged["gold"])
    ].copy()

    print(f"{rag_method}: {len(helped)} cases where RAG helped")

    for _, row in helped.head(n).iterrows():
        print("\n" + "=" * 100)
        print("QID:", row["qid"])
        print("Question:", row["question"])
        print("Gold:", row["gold"])
        print("No-RAG Pred:", row["no_rag_pred"])
        print(f"{rag_method} Pred:", row["pred"])
        print("Retrieved PMIDs:", row["retrieved_pmids"])

show_rag_helped("semantic_rag", n=5)

semantic_rag: 223 cases where RAG helped

QID: 22540518
Question: Is micro-computed tomography reliable to determine the microstructure of the maxillary alveolar bone?
Gold: yes
No-RAG Pred: maybe
semantic_rag Pred: yes
Retrieved PMIDs: ['22540518', '38606694']

QID: 20306735
Question: Fulfilling human resources development goal in West Africa: can the training of ophthalmologist diplomates be improved?
Gold: yes
No-RAG Pred: maybe
semantic_rag Pred: yes
Retrieved PMIDs: ['20306735']

QID: 21394762
Question: Is pelvic pain associated with defecatory symptoms in women with pelvic organ prolapse?
Gold: yes
No-RAG Pred: unknown
semantic_rag Pred: yes
Retrieved PMIDs: ['21394762', '30576003', '35550798', '40498384', '23575699', '25874639']

QID: 23455575
Question: Globulomaxillary cysts--do they really exist?
Gold: no
No-RAG Pred: maybe
semantic_rag Pred: no
Retrieved PMIDs: ['23455575']

QID: 18239988
Question: Differentiation of nonalcoholic from alcoholic steatohepatitis: are routine la

In [ ]:
# =========================
# CASES WHERE RAG HURT
# =========================

def show_rag_hurt(rag_method="semantic_rag", n=5):
    no_rag = pred_df[pred_df["method"] == "no_rag"][["qid", "pred"]].rename(
        columns={"pred": "no_rag_pred"}
    )

    rag = pred_df[pred_df["method"] == rag_method].copy()

    merged = rag.merge(no_rag, on="qid", how="inner")

    hurt = merged[
        (merged["no_rag_pred"] == merged["gold"]) &
        (merged["pred"] != merged["gold"])
    ].copy()

    print(f"{rag_method}: {len(hurt)} cases where RAG hurt")

    for _, row in hurt.head(n).iterrows():
        print("\n" + "=" * 100)
        print("QID:", row["qid"])
        print("Question:", row["question"])
        print("Gold:", row["gold"])
        print("No-RAG Pred:", row["no_rag_pred"])
        print(f"{rag_method} Pred:", row["pred"])
        print("Retrieved PMIDs:", row["retrieved_pmids"])
        print("Error Type:", row["error_type"])

show_rag_hurt("semantic_rag", n=5)

semantic_rag: 37 cases where RAG hurt

QID: 16392897
Question: BCRABL transcript detection by quantitative real-time PCR : are correlated results possible from homebrew assays?
Gold: maybe
No-RAG Pred: maybe
semantic_rag Pred: yes
Retrieved PMIDs: ['16392897']
Error Type: generation_failure

QID: 20337202
Question: Continuation of pregnancy after antenatal corticosteroid administration: opportunity for rescue?
Gold: maybe
No-RAG Pred: maybe
semantic_rag Pred: yes
Retrieved PMIDs: ['20337202']
Error Type: generation_failure

QID: 22227642
Question: Can we measure mesopic pupil size with the cobalt blue light slit-lamp biomicroscopy method?
Gold: no
No-RAG Pred: no
semantic_rag Pred: yes
Retrieved PMIDs: ['22227642']
Error Type: generation_failure

QID: 12407608
Question: Does ultrasound imaging before puncture facilitate internal jugular vein cannulation?
Gold: maybe
No-RAG Pred: maybe
semantic_rag Pred: yes
Retrieved PMIDs: ['12407608', '1541105', '16130056', '18942245', '16409336', '7

In [9]:
import nbformat

file_path = "/content/PubMedQA_CBR_RAG_Gemma_Evaluation.ipynb"

nb = nbformat.read(file_path, as_version=4)

# Remove problematic widget metadata
if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

nbformat.write(nb, file_path)

print("Notebook cleaned and fixed!")

FileNotFoundError: [Errno 2] No such file or directory: '/content/PubMedQA_CBR_RAG_Gemma_Evaluation.ipynb'